# Speech Commands Audio Similarity using YAMNet

## Objective
Load audio clips from the TensorFlow Speech Commands dataset, extract embeddings using YAMNet, compute similarity and distance matrices, and visualize the results.

**Dataset:** https://www.tensorflow.org/datasets/catalog/speech_commands

**Model:** https://tfhub.dev/google/yamnet/1


In [ ]:
!pip -q install tensorflow tensorflow-datasets tensorflow-hub librosa seaborn

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances


In [ ]:
dataset = tfds.load("speech_commands", split="train", as_supervised=True)
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")


In [ ]:
audio_list=[]
labels=[]

for audio,label in dataset.take(12):
    audio_list.append(audio.numpy().astype(np.float32))
    labels.append(int(label.numpy()))

print("Loaded",len(audio_list),"audio clips")


In [ ]:
builder=tfds.builder("speech_commands")
builder.download_and_prepare()

label_names=builder.info.features["label"].names
label_text=[label_names[i] for i in labels]

print(label_text)


In [ ]:
embeddings=[]

for waveform in audio_list:
    scores,embedding,spectrogram=yamnet(tf.convert_to_tensor(waveform))
    embeddings.append(tf.reduce_mean(embedding,axis=0).numpy())

embeddings=np.array(embeddings)
print(embeddings.shape)


In [ ]:
similarity_matrix=cosine_similarity(embeddings)
distance_matrix=pairwise_distances(embeddings,metric="euclidean")


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(similarity_matrix,
            annot=True,
            cmap="viridis",
            xticklabels=label_text,
            yticklabels=label_text)
plt.title("Cosine Similarity Heatmap")
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(distance_matrix,
            annot=True,
            cmap="magma",
            xticklabels=label_text,
            yticklabels=label_text)
plt.title("Euclidean Distance Matrix")
plt.show()


## Observations

1. Similar audio clips have higher cosine similarity.
2. The diagonal of the similarity matrix is 1.
3. Euclidean distance is 0 on the diagonal.
4. YAMNet produces 1024-dimensional embeddings suitable for audio similarity analysis.
